# Step 19 — level 2, a shared gene list: sites agree on which genes, and nothing else

**Data type: RNA_array** (GSE65391). **Reads:** `step10_site_{A,B,C}.rds`, `step11_features.rds`.
**Writes:** `step19_site_{A,B,C}.rds`.

The cheapest thing federation can share is **a list of gene names**: the 1,000 genes chosen in step 11
from pooled per-gene variances. Each site then clusters alone, exactly as in step 18, but on the shared
genes. Centring, components, consensus, k and centroids all stay local.

In the proteomics study this level delivered almost all of the benefit. This notebook tests whether
that holds here.

In [1]:
source("../src/paths.R")
source("../src/endotypes.R")
genes <- readRDS(art("step04_genes.rds")); expr <- genes$keep[genes$expressed]
shared <- readRDS(art("step11_features.rds"))$genes
fit_site <- function(E, core, gene_list, seed) {
  set.seed(seed)
  centre <- rowMeans(E[gene_list, core])
  Z  <- t(E[gene_list, core] - centre)
  S  <- prcomp(Z)$x[, 1:10]
  C  <- lapply(setNames(2:6, 2:6), function(k) consensus_matrix(S, k, n_resamples = 100))
  pt <- data.frame(k = 2:6, pac = sapply(C, pac))
  k  <- choose_k(pt)
  lab <- labels_from_consensus(C[[as.character(k)]], k)
  centroids <- t(sapply(levels(lab), function(e) colMeans(Z[lab == e, , drop = FALSE])))
  list(genes = gene_list, centre = centre, centroids = centroids, k = k, pac = pt, labels = lab)
}
core_of <- function(m) { d <- m[m$split == "discovery", ]; d <- d[order(d$subject, d$visit), ]
                         rownames(d)[!duplicated(d$subject)] }
for (i in seq_along(SITES)) {
  s <- SITES[i]; d <- readRDS(site_file("10", s))
  model <- fit_site(d$E, core_of(d$meta), shared, seed = SEED + 190L + i)
  saveRDS(model, site_file("19", s))
  cat(sprintf("site %s with shared genes: k = %d, PAC %.3f, sizes %s\n", s, model$k,
              model$pac$pac[model$pac$k == model$k], paste(table(model$labels), collapse = "/")))
}

site A with shared genes: k = 6, PAC 0.324, sizes 10/8/8/5/5/1
site B with shared genes: k = 5, PAC 0.186, sizes 11/10/9/5/2
site C with shared genes: k = 2, PAC 0.075, sizes 21/15


In [2]:
sapply(SITES, function(s) readRDS(site_file("19", s))$pac$pac)

A,B,C
0.3528529,0.5390390,0.07460317
0.3993994,0.5105105,0.39841270
0.3873874,0.2822823,0.26507937
0.3513514,0.1861862,0.32063492
0.3243243,0.2312312,0.32380952


## Findings

**A shared gene list is not enough here.** On the shared 1,000 genes, the sites make the same choices
as alone: A takes k = 6, B k = 5, C k = 2. PAC moves a little in either direction. The shared list
removes one source of disagreement (which genes), but the main problem is the small number of
patients per site, and a gene list does not add patients.

This differs from the proteomics study, where the shared list gave almost all of the benefit. There,
each cohort had about 87 patients; here a site has about 37.